# Maintenance Knowledge Base and Hybrid RAG Baseline

This notebook creates a small, versioned synthetic maintenance knowledge base
and builds a local retrieval baseline for the RCA workflow.

```text
telemetry + vision evidence
  -> retrieval query
  -> semantic similarity + exact keyword matching
  -> cited maintenance passages
  -> RCA reasoning layer
```

No LLM fine-tuning and no external API are required. The documents are clearly
marked synthetic and are suitable for demonstrating retrieval, citations, and
the end-to-end contract. They are not manufacturer instructions.


## 1. Environment and Paths


In [6]:
# Run only if missing. PyTorch must remain the AMD ROCm build.
%pip install "numpy<2.3" "pandas<3" "scikit-learn<2" sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 2.7 MB/s  0:00:0036m-:--:--

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
from pathlib import Path
from datetime import date
import json
import re
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

SEED = 42
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SEMANTIC_WEIGHT = 0.75
KEYWORD_WEIGHT = 0.25
TOP_K = 5

KNOWLEDGE_ROOT = Path("/workspace/notebooks/data/knowledge")
ARTIFACT_ROOT = Path("/workspace/notebooks/knowledge/artifacts/rag")
DOCUMENT_PATH = KNOWLEDGE_ROOT / "maintenance_knowledge.jsonl"
INDEX_PATH = ARTIFACT_ROOT / "maintenance_embeddings.npy"
SOURCE_STORE_PATH = ARTIFACT_ROOT / "source_store.jsonl"
METADATA_PATH = ARTIFACT_ROOT / "metadata.json"

KNOWLEDGE_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Knowledge root:", KNOWLEDGE_ROOT)
print("Artifacts:", ARTIFACT_ROOT)
print("Embedding device:", DEVICE)
print("ROCm/HIP version:", torch.version.hip)


Knowledge root: /workspace/notebooks/data/knowledge
Artifacts: /workspace/notebooks/knowledge/artifacts/rag
Embedding device: cuda
ROCm/HIP version: 7.0.51831-a3e329ad8


## 2. Create the Versioned Synthetic Knowledge Pack


In [8]:
documents = [
    {
        "document_id": "SOP-SAFE-001",
        "title": "Abnormal Robot Joint Safe Response",
        "document_type": "sop",
        "asset_type": "robot_joint",
        "revision": "1.0",
        "effective_date": "2026-06-15",
        "section": "Immediate response",
        "text": (
            "When telemetry indicates critical degradation or an inspection "
            "shows a structural surface anomaly, place the robot in a safe "
            "state, isolate motion energy, and prevent automatic restart. "
            "A qualified technician must inspect the affected joint before "
            "return to service. Do not infer component replacement from an "
            "anomaly score alone."
        ),
        "tags": ["critical", "shutdown", "inspection", "robot joint", "safety"],
        "source_status": "synthetic",
    },
    {
        "document_id": "SOP-GBX-004",
        "title": "Gearbox Degradation Inspection",
        "document_type": "sop",
        "asset_type": "gearbox",
        "revision": "1.1",
        "effective_date": "2026-06-15",
        "section": "Inspection",
        "text": (
            "Low remaining useful life together with rising vibration or "
            "temperature can indicate gearbox or bearing degradation. Inspect "
            "lubrication level, gear backlash, bearing play, fastener torque, "
            "and abnormal noise. Confirm the cause before replacing parts."
        ),
        "tags": ["gearbox", "bearing", "vibration", "temperature", "low RUL"],
        "source_status": "synthetic",
    },
    {
        "document_id": "GUIDE-BRG-002",
        "title": "Bearing Wear Troubleshooting",
        "document_type": "troubleshooting",
        "asset_type": "bearing",
        "revision": "1.0",
        "effective_date": "2026-06-15",
        "section": "Symptoms and checks",
        "text": (
            "Bearing wear commonly presents as increasing vibration, heat, "
            "rough motion, or periodic noise. Check alignment, lubrication, "
            "radial play, mounting condition, and load history. Similar "
            "symptoms can also result from misalignment or loose fasteners."
        ),
        "tags": ["bearing wear", "vibration", "heat", "misalignment"],
        "source_status": "synthetic",
    },
    {
        "document_id": "GUIDE-CRK-003",
        "title": "Structural Crack Inspection",
        "document_type": "inspection_guide",
        "asset_type": "robot_joint",
        "revision": "1.0",
        "effective_date": "2026-06-15",
        "section": "Surface discontinuity",
        "text": (
            "A suspected crack or linear surface discontinuity requires close "
            "visual inspection and an appropriate nondestructive examination. "
            "Remove the asset from high-load service until the indication is "
            "confirmed. Do not classify heatmap localization as crack proof."
        ),
        "tags": ["crack", "surface anomaly", "NDT", "high load"],
        "source_status": "synthetic",
    },
    {
        "document_id": "GUIDE-LUB-005",
        "title": "Oil Leak and Lubrication Loss",
        "document_type": "troubleshooting",
        "asset_type": "gearbox",
        "revision": "1.0",
        "effective_date": "2026-06-15",
        "section": "Leak response",
        "text": (
            "Dark wet residue below a seal or joint can indicate lubricant "
            "leakage. Inspect seals, fittings, drain points, lubricant level, "
            "and contamination. Clean the area before confirming leak growth. "
            "Lubrication loss can accelerate bearing and gear wear."
        ),
        "tags": ["oil leak", "lubrication", "seal", "gearbox"],
        "source_status": "synthetic",
    },
    {
        "document_id": "GUIDE-COR-006",
        "title": "Corrosion Assessment",
        "document_type": "inspection_guide",
        "asset_type": "robot_joint",
        "revision": "1.0",
        "effective_date": "2026-06-15",
        "section": "Surface condition",
        "text": (
            "Localized discoloration, pitting, or oxidation may indicate "
            "corrosion. Inspect coating damage, moisture exposure, electrical "
            "connectors, and material loss. Surface appearance alone does not "
            "establish structural severity."
        ),
        "tags": ["corrosion", "oxidation", "pitting", "moisture"],
        "source_status": "synthetic",
    },
    {
        "document_id": "GUIDE-HEAT-007",
        "title": "Overheating and Discoloration",
        "document_type": "troubleshooting",
        "asset_type": "motor",
        "revision": "1.0",
        "effective_date": "2026-06-15",
        "section": "Thermal symptoms",
        "text": (
            "Thermal discoloration with elevated motor current or temperature "
            "can indicate overload, friction, cooling restriction, or "
            "electrical resistance. Verify temperature with an independent "
            "instrument and inspect load, cooling paths, and connections."
        ),
        "tags": ["overheating", "temperature", "motor current", "overload"],
        "source_status": "synthetic",
    },
    {
        "document_id": "GUIDE-WEAR-008",
        "title": "Mechanical Surface Wear",
        "document_type": "inspection_guide",
        "asset_type": "robot_joint",
        "revision": "1.0",
        "effective_date": "2026-06-15",
        "section": "Wear indicators",
        "text": (
            "Polishing, scoring, abrasion, or material removal near a moving "
            "interface may indicate mechanical wear. Inspect alignment, "
            "clearance, lubrication, foreign material, and contact pattern."
        ),
        "tags": ["wear", "abrasion", "alignment", "clearance"],
        "source_status": "synthetic",
    },
    {
        "document_id": "ERR-E204",
        "title": "E204 Excessive Vibration Event",
        "document_type": "error_code",
        "asset_type": "robot_joint",
        "revision": "1.0",
        "effective_date": "2026-06-15",
        "section": "Code definition",
        "text": (
            "E204 denotes vibration above the configured operating envelope. "
            "Possible causes include bearing degradation, gearbox damage, "
            "misalignment, loose mounting, or abnormal load. Inspect evidence "
            "from other modalities before assigning a root cause."
        ),
        "tags": ["E204", "vibration", "bearing", "gearbox", "misalignment"],
        "source_status": "synthetic",
    },
    {
        "document_id": "ERR-E310",
        "title": "E310 Motor Temperature High",
        "document_type": "error_code",
        "asset_type": "motor",
        "revision": "1.0",
        "effective_date": "2026-06-15",
        "section": "Code definition",
        "text": (
            "E310 denotes motor temperature above its configured limit. "
            "Investigate overload, restricted cooling, excessive friction, "
            "ambient temperature, and electrical connection resistance."
        ),
        "tags": ["E310", "temperature", "motor", "overload", "cooling"],
        "source_status": "synthetic",
    },
    {
        "document_id": "CHECK-ALIGN-009",
        "title": "Joint Alignment Inspection Checklist",
        "document_type": "checklist",
        "asset_type": "robot_joint",
        "revision": "1.0",
        "effective_date": "2026-06-15",
        "section": "Alignment",
        "text": (
            "For suspected misalignment, verify mounting surfaces, coupling "
            "offset, shaft runout, fastener condition, and repeatability under "
            "low load. Misalignment can produce periodic vibration and wear."
        ),
        "tags": ["misalignment", "coupling", "runout", "vibration"],
        "source_status": "synthetic",
    },
    {
        "document_id": "GUIDE-SENSOR-010",
        "title": "Sensor Drift Validation",
        "document_type": "troubleshooting",
        "asset_type": "sensor",
        "revision": "1.0",
        "effective_date": "2026-06-15",
        "section": "Signal validation",
        "text": (
            "A slowly changing signal without corroborating physical evidence "
            "may indicate sensor drift. Compare redundant measurements, check "
            "calibration history, inspect wiring, and validate with a reference "
            "instrument before declaring machine degradation."
        ),
        "tags": ["sensor drift", "calibration", "wiring", "reference"],
        "source_status": "synthetic",
    },
]

required_fields = {
    "document_id", "title", "document_type", "asset_type", "revision",
    "effective_date", "section", "text", "tags", "source_status",
}
ids = [document["document_id"] for document in documents]
if len(ids) != len(set(ids)):
    raise ValueError("Knowledge document IDs must be unique.")

for document in documents:
    missing = required_fields - set(document)
    if missing:
        raise ValueError(
            f"{document.get('document_id')} missing fields: {missing}"
        )
    date.fromisoformat(document["effective_date"])
    if document["source_status"] not in {"synthetic", "licensed", "internal"}:
        raise ValueError("Invalid source_status.")

with DOCUMENT_PATH.open("w", encoding="utf-8") as handle:
    for document in documents:
        handle.write(json.dumps(document, ensure_ascii=False) + "\n")

print("Saved documents:", DOCUMENT_PATH)
print("Document count:", len(documents))
display(pd.DataFrame(documents)[
    ["document_id", "document_type", "asset_type", "revision", "source_status"]
])


Saved documents: /workspace/notebooks/data/knowledge/maintenance_knowledge.jsonl
Document count: 12


,document_id,document_type,asset_type,revision,source_status
0,SOP-SAFE-001,sop,robot_joint,1.0,synthetic
1,SOP-GBX-004,sop,gearbox,1.1,synthetic
2,GUIDE-BRG-002,troubleshooting,bearing,1.0,synthetic
3,GUIDE-CRK-003,inspection_guide,robot_joint,1.0,synthetic
4,GUIDE-LUB-005,troubleshooting,gearbox,1.0,synthetic
5,GUIDE-COR-006,inspection_guide,robot_joint,1.0,synthetic
6,GUIDE-HEAT-007,troubleshooting,motor,1.0,synthetic
7,GUIDE-WEAR-008,inspection_guide,robot_joint,1.0,synthetic
8,ERR-E204,error_code,robot_joint,1.0,synthetic
9,ERR-E310,error_code,motor,1.0,synthetic


## 3. Build the Embedding Index


In [9]:
def embedding_text(document):
    return " | ".join([
        document["title"],
        document["asset_type"],
        document["section"],
        " ".join(document["tags"]),
        document["text"],
    ])


started = time.perf_counter()
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL,
    device=DEVICE,
)
corpus = [embedding_text(document) for document in documents]
embeddings = embedding_model.encode(
    corpus,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
)

np.save(INDEX_PATH, embeddings)
with SOURCE_STORE_PATH.open("w", encoding="utf-8") as handle:
    for document in documents:
        handle.write(json.dumps(document, ensure_ascii=False) + "\n")

print("Embedding shape:", embeddings.shape)
print(f"Index build time: {time.perf_counter() - started:.2f}s")
print("Saved index:", INDEX_PATH)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (12, 384)
Index build time: 21.99s
Saved index: /workspace/notebooks/knowledge/artifacts/rag/maintenance_embeddings.npy


## 4. Hybrid Semantic and Keyword Retrieval


In [10]:
def tokens(text):
    return set(re.findall(r"[a-z0-9]+", text.lower()))


document_tokens = [
    tokens(embedding_text(document))
    for document in documents
]


def keyword_score(query, document_index):
    query_tokens = tokens(query)
    if not query_tokens:
        return 0.0
    overlap = query_tokens & document_tokens[document_index]
    base_score = len(overlap) / len(query_tokens)

    # Exact error codes receive a strong deterministic boost.
    query_codes = set(re.findall(r"\bE\d{3}\b", query.upper()))
    document_id = documents[document_index]["document_id"].upper()
    code_boost = 1.0 if any(code in document_id for code in query_codes) else 0.0
    return min(1.0, base_score + code_boost)


def retrieve(query, top_k=TOP_K, filters=None):
    started = time.perf_counter()
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    semantic_scores = cosine_similarity(
        query_embedding, embeddings
    )[0]

    results = []
    for index, document in enumerate(documents):
        if filters:
            if any(
                document.get(key) != value
                for key, value in filters.items()
            ):
                continue
        exact_score = keyword_score(query, index)
        hybrid_score = (
            SEMANTIC_WEIGHT * max(float(semantic_scores[index]), 0.0)
            + KEYWORD_WEIGHT * exact_score
        )
        results.append({
            "document_id": document["document_id"],
            "title": document["title"],
            "section": document["section"],
            "revision": document["revision"],
            "source_status": document["source_status"],
            "semantic_score": float(semantic_scores[index]),
            "keyword_score": float(exact_score),
            "score": float(hybrid_score),
            "text": document["text"],
        })

    results.sort(key=lambda item: item["score"], reverse=True)
    return {
        "query": query,
        "latency_ms": (time.perf_counter() - started) * 1000,
        "results": results[:top_k],
    }


## 5. Retrieve for the Current Multimodal Demo


In [11]:
telemetry_contract = {
    "predicted_rul": 12.0,
    "failure_risk": 0.91,
    "severity": "critical",
    "evidence": {
        "failure_horizon_cycles": 30,
    },
}

vision_contract = {
    "is_anomaly": True,
    "predicted_fault": "visual_anomaly",
    "severity": "unknown",
    "location": "dinov2_patch_heatmap",
}

event_codes = ["E204"]

query = (
    "robot joint critical degradation; "
    f"predicted RUL {telemetry_contract['predicted_rul']} cycles; "
    f"failure risk {telemetry_contract['failure_risk']}; "
    "visual surface anomaly localized by heatmap; "
    f"event codes {' '.join(event_codes)}; "
    "safe response inspection gearbox bearing vibration"
)

rag_output = retrieve(query, top_k=5)
print("Query:", rag_output["query"])
print(f"Latency: {rag_output['latency_ms']:.2f} ms")
display(pd.DataFrame(rag_output["results"])[
    [
        "document_id", "title", "revision", "source_status",
        "semantic_score", "keyword_score", "score",
    ]
])


Query: robot joint critical degradation; predicted RUL 12.0 cycles; failure risk 0.91; visual surface anomaly localized by heatmap; event codes E204; safe response inspection gearbox bearing vibration
Latency: 11.24 ms


,document_id,title,revision,source_status,semantic_score,keyword_score,score
0,ERR-E204,E204 Excessive Vibration Event,1.0,synthetic,0.560918,1.000000,0.670688
1,SOP-GBX-004,Gearbox Degradation Inspection,1.1,synthetic,0.670494,0.222222,0.558426
2,SOP-SAFE-001,Abnormal Robot Joint Safe Response,1.0,synthetic,0.591326,0.333333,0.526828
3,CHECK-ALIGN-009,Joint Alignment Inspection Checklist,1.0,synthetic,0.548033,0.148148,0.448061
4,GUIDE-CRK-003,Structural Crack Inspection,1.0,synthetic,0.507211,0.259259,0.445223


## 6. Retrieval Evaluation


In [12]:
evaluation_cases = [
    {
        "query": "E204 excessive vibration robot joint",
        "relevant_ids": {"ERR-E204"},
    },
    {
        "query": "low RUL vibration heat inspect gearbox bearing",
        "relevant_ids": {"SOP-GBX-004", "GUIDE-BRG-002"},
    },
    {
        "query": "dark oil residue below gearbox seal",
        "relevant_ids": {"GUIDE-LUB-005"},
    },
    {
        "query": "linear surface anomaly possible crack NDT",
        "relevant_ids": {"GUIDE-CRK-003"},
    },
    {
        "query": "motor hot elevated current restricted cooling",
        "relevant_ids": {"GUIDE-HEAT-007", "ERR-E310"},
    },
    {
        "query": "critical visual anomaly safe stop inspect robot joint",
        "relevant_ids": {"SOP-SAFE-001"},
    },
]

evaluation_rows = []
for case_index, case in enumerate(evaluation_cases, start=1):
    output = retrieve(case["query"], top_k=5)
    ranked_ids = [
        result["document_id"] for result in output["results"]
    ]
    relevant_ranks = [
        rank
        for rank, document_id in enumerate(ranked_ids, start=1)
        if document_id in case["relevant_ids"]
    ]
    evaluation_rows.append({
        "case": case_index,
        "query": case["query"],
        "expected": sorted(case["relevant_ids"]),
        "top_1": ranked_ids[0],
        "recall_at_1": int(bool(relevant_ranks and min(relevant_ranks) <= 1)),
        "recall_at_3": int(bool(relevant_ranks and min(relevant_ranks) <= 3)),
        "recall_at_5": int(bool(relevant_ranks and min(relevant_ranks) <= 5)),
        "reciprocal_rank": (
            1.0 / min(relevant_ranks) if relevant_ranks else 0.0
        ),
        "latency_ms": output["latency_ms"],
    })

evaluation = pd.DataFrame(evaluation_rows)
metrics = {
    "recall_at_1": float(evaluation["recall_at_1"].mean()),
    "recall_at_3": float(evaluation["recall_at_3"].mean()),
    "recall_at_5": float(evaluation["recall_at_5"].mean()),
    "mean_reciprocal_rank": float(
        evaluation["reciprocal_rank"].mean()
    ),
    "mean_latency_ms": float(evaluation["latency_ms"].mean()),
}

display(evaluation)
display(pd.DataFrame([metrics]))


,case,query,expected,top_1,recall_at_1,recall_at_3,recall_at_5,reciprocal_rank,latency_ms
0,1,E204 excessive vibration robot joint,[ERR-E204],ERR-E204,1,1,1,1.0,6.554445
1,2,low RUL vibration heat inspect gearbox bearing,"[GUIDE-BRG-002, SOP-GBX-004]",SOP-GBX-004,1,1,1,1.0,5.374546
2,3,dark oil residue below gearbox seal,[GUIDE-LUB-005],GUIDE-LUB-005,1,1,1,1.0,5.297316
3,4,linear surface anomaly possible crack NDT,[GUIDE-CRK-003],GUIDE-CRK-003,1,1,1,1.0,5.159822
4,5,motor hot elevated current restricted cooling,"[ERR-E310, GUIDE-HEAT-007]",GUIDE-HEAT-007,1,1,1,1.0,5.146436
5,6,critical visual anomaly safe stop inspect robo...,[SOP-SAFE-001],SOP-SAFE-001,1,1,1,1.0,5.142134


,recall_at_1,recall_at_3,recall_at_5,mean_reciprocal_rank,mean_latency_ms
0,1.0,1.0,1.0,1.0,5.445783


## 7. Validate Citations and Save the RAG Contract


In [13]:
known_ids = {document["document_id"] for document in documents}
retrieved_ids = {
    result["document_id"] for result in rag_output["results"]
}
if not retrieved_ids.issubset(known_ids):
    raise ValueError("Retrieval returned an unknown document ID.")

rag_contract = {
    "query_id": "RAG-SYNTH-MECH-001",
    "scenario_id": "SYNTH-MECH-001",
    "query": rag_output["query"],
    "latency_ms": rag_output["latency_ms"],
    "results": rag_output["results"],
    "limitations": [
        "The maintenance documents are synthetic demonstration content.",
        "Retrieved passages are advisory and not manufacturer procedures.",
        "The reasoning layer must cite only document IDs in these results.",
    ],
}

metadata = {
    "embedding_model": EMBEDDING_MODEL,
    "device": DEVICE,
    "document_count": len(documents),
    "embedding_dimension": int(embeddings.shape[1]),
    "semantic_weight": SEMANTIC_WEIGHT,
    "keyword_weight": KEYWORD_WEIGHT,
    "evaluation_metrics": metrics,
    "source_status": "synthetic",
}

(ARTIFACT_ROOT / "rag_contract_example.json").write_text(
    json.dumps(rag_contract, indent=2),
    encoding="utf-8",
)
METADATA_PATH.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)
evaluation.to_csv(
    ARTIFACT_ROOT / "retrieval_evaluation.csv",
    index=False,
)

print("Saved RAG contract:", ARTIFACT_ROOT / "rag_contract_example.json")
print("Saved metadata:", METADATA_PATH)
print("Citation IDs:", sorted(retrieved_ids))


Saved RAG contract: /workspace/notebooks/knowledge/artifacts/rag/rag_contract_example.json
Saved metadata: /workspace/notebooks/knowledge/artifacts/rag/metadata.json
Citation IDs: ['CHECK-ALIGN-009', 'ERR-E204', 'GUIDE-CRK-003', 'SOP-GBX-004', 'SOP-SAFE-001']


## Completion Criteria

- Synthetic maintenance documents pass metadata validation.
- Embeddings are created using the AMD ROCm device when available.
- Hybrid retrieval returns document IDs, revisions, scores, and passages.
- Exact error-code matching is supported.
- Recall@1, Recall@3, Recall@5, MRR, and latency are reported.
- Every citation maps to a real retrieved document.
- The RAG contract is ready for evidence fusion and RCA.

The next notebook will combine the telemetry, vision, and RAG contracts into a
structured rule-based RCA baseline, followed by an optional pretrained
reasoning model.
